## **Techno-economic Assessment Sample - Week 05**

Quick-and-dirty demonstration of how to develop a TEA analysis from sample data/equipment.
The process is a simple "methanol synthesis". There is no real-life application, rather it serves as an examination of code.

*All data used is taken from OpenPyTea's tutorial videos and notebooks.*

### **Step #1. Import OpenPyTEA**

In [1]:
from openpytea.equipment import Equipment
from openpytea.plant import Plant
from openpytea.analysis import *

### **Step #2. Implement Equipment Objects**

There are two ways to implement equipment objects.
| Method | When to use |
|---|---|
| Database lookup (`param`) | Equipment size is known; the type exists in the built-in database |
| Direct cost (`purchased_cost` + `cost_year`) | A vendor quote or literature value is available |

For the built-in database, see cost_correlations.csv and cepci_values.csv.
In the first steps of the technical implementation, we use the database lookup method to get initial values and figure out additional parameters to add.

Once we get **more detailed equipment information - year of model, vendor, literature review values, etc.** (from Group 3), we fine-tune the Equipment objects with the Direct cost method.

In [2]:
compressor = Equipment(
    name="COMP-1",
    param=100,
    process_type="Fluids",
    category="Compressors, fans, & blowers",
    type="Compressor, centrifugal",
    material="Carbon steel",
)

heat_exchanger = Equipment(
    name="HX-1",
    param=35,
    process_type="Fluids",
    category="Heat Exchangers",
    type="U-tube shell & tube",
    material="304 stainless steel",
)

reactor = Equipment(
    name="REACT-1",
    param=0,
    process_type="Fluids",
    category="Reactors",
    purchased_cost=1_500_000,
    cost_year=2019,
)

pump = Equipment(
    name="PUMP-1",
    param=10,
    process_type="Fluids",
    category="Pumps",
    type="Centrifugal",
    material="Carbon steel",
)

equipment_list = [compressor, heat_exchanger, reactor, pump]
print(compressor) # Example

Name=COMP-1, Category=Compressors, fans, & blowers, Sub-type=Compressor, centrifugal, Material=Carbon steel, Process Type=Fluids, Parameter=100, Number of units=1, Purchased Cost=1302801.2178536279, Direct Cost=4168963.8971316093)


Note how, after we provide the necessary parameters to the Equipment object, **we can see that the framework has calculated direct costs and purchase costs of the unit for us**.

- **Direct Cost**: An expense directly traceable to the unit (e.g. construction, raw material, labour, operation)
- **Purchased Cost**: How much it cost to buy the unit (not operate it)

### **Step #3. Create a Processing Plant**

The `Plant` constructor takes a single Python dictionary.
Much of these are to be figured out as Group 3 conducts research on factors such as equipment and location rates. For now, we've used some example data from the framework's tutorial.

In [3]:
config = {
    # --- Plant identity ---
    "plant_name": "Sample Methanol Synthesis",     # A label for the Plant
    "process_type": "Fluids",               # The type of process ("Fluids", "Solids", or "Mixed")

    # --- Plant location ---
    "country": "Australia",             # country where the plant is located                
    "region": "Victoria",                 # region within the country  

    # --- Equipment ---
    "equipment": equipment_list,            # list of Equipment objects in the plant

    # --- Economic parameters ---
    "interest_rate":     0.08,              # 8 % discount rate
    "project_lifetime":  20,                # years
    "plant_utilization": 0.90,              # 90 % capacity factor
    "tax_rate":          0.25,              # 25 % corporate tax

    # --- Labour ---
    "operator_hourly_rate": {
        "rate": 38.0,     # USD/hr (mean)
    },

    # --- Products ---
    # production: kg/day    price: unit price
    "plant_products": {
        "methanol": {
            "production": 45_000,  # kg/day
            "price":      1.0,     # USD/kg
        },
        "carbon_dioxide": {
            "production": 10_000,  # kg/day
            "price":      0.6,     # USD/kg
        }
    },

    # --- Variable operating costs ---
    # consumption: daily amount    price: unit cost
    "variable_opex_inputs": {
        "natural_gas": {
            "consumption": 35_000,  # kg/day
            "price":       0.4,     # USD/kg
        },
        "process_water": {
            "consumption": 20_000,  # kg/day
            "price":       0.002,   # USD/kg
        },
        "cooling_water": {
            "consumption": 5_000,  # kg/day
            "price":       0.0005, # USD/kg
        },
        "electricity": {
            "consumption": 10_000, # kWh/day
            "price":       0.08,   # USD/kWh
        }
    },

    # --- Depreciation: the decrease in efficiency for the plant ---
    "depreciation": {
        "method": "straight_line",
        "life": 12,
        "salvage_fraction": 0.05,
        "service_start_year": 2
    }
}

In [4]:
plant = Plant(config)

### **Step 4 — Run the Calculations**

Time to conduct some high-level TEA calculations for our plant!
We will begin with capital cost breakdowns.

In [5]:
# Capital cost breakdown (set print_results=True for a detailed table)
plant.calculate_fixed_capital(print_results=True)

Capital cost estimation
ISBL: 12,941,259.48 USD
OSBL: 3,882,377.84 USD
Design and engineering: 5,047,091.20 USD
Contingency: 1,682,363.73 USD
Fixed capital investment: 23,553,092.25 USD


- **ISBL**: Inside Battery Limits, representing the purchase and installation of the plant & equipment
- **OSBL**: Outside Battery Limits, representing off-site development (e.g. water and electricity needed to operate equipment, labour costs).  

In [6]:
# ISBL breakdown (set print_results=True for a detailed table)
plant.calculate_isbl(print_results=True)

ISBL cost estimation
  - COMP-1: 4,168,963.90 USD
  - HX-1: 173,003.42 USD
  - REACT-1: 6,320,987.65 USD
  - PUMP-1: 32,300.79 USD
Total ISBL: 12,941,259.48 USD


In [7]:
# Annual operating costs
plant.calculate_variable_opex(print_results=True)

Variable production costs estimation
  - Natural gas: 4,599,000.00 USD per year
  - Process water: 13,140.00 USD per year
  - Cooling water: 821.25 USD per year
  - Electricity: 262,800.00 USD per year
Total Variable OPEX: 4,875,761.25USD per year


In [8]:
plant.calculate_fixed_opex(print_results=True)

Fixed production costs estimation
Operating labor costs: 893,760.00 USD per year
Supervision costs: 223,440.00 USD per year
Direct salary overhead: 558,600.00 USD per year
Laboratory charges: 89,376.00 USD per year
Maintenance costs: 647,062.97 USD per year
Taxes and insurance costs: 194,118.89 USD per year
Rent of land costs: 252,354.56 USD per year
Environmental charges: 168,236.37 USD per year
Operating supplies: 116,471.34 USD per year
General plant overhead: 1,089,270.00 USD per year
Interest on working capital: 282,637.11 USD per year
Patents and royalties: 201,958.89 USD per year
Distribution and selling costs: 201,958.89 USD per year
R&D costs: 302,938.34 USD per year
Fixed OPEX: 5,222,183.36 USD per year


In [9]:
# Annual revenue
plant.calculate_revenue(print_results=True)

Revenue estimation
  - Methanol: 14,782,500.00 USD per year
  - Carbon dioxide: 1,971,000.00 USD per year
Total Revenue: 16,753,500.00 USD per year


Now, we conduct some calculations for **financial metrics, a.k.a. how profitable our plant is as an investment.**

In [10]:
# Key financial metrics
plant.calculate_levelized_cost(print_results=True)
plant.calculate_roi(print_results=True)
plant.calculate_irr(print_results=True)
plant.calculate_payback_time(print_results=True)
plant.calculate_npv(print_results=True)

Levelized cost: 0.871 USD/unit
Return of investment: 14.57%
Internal Rate of Return: 10.26%
Payback time: 4.87 years
Year | Present Value [USD] | Cumulative NPV [USD]
-------------------------------------------
   1 |  -11,377,880.59 |  -11,377,880.59
   2 |  -16,592,968.72 |  -27,970,849.31
   3 |   -5,048,265.74 |  -33,019,115.05
   4 |    3,145,933.38 |  -29,873,181.67
   5 |    4,118,691.07 |  -25,754,490.60
   6 |    3,439,353.33 |  -22,315,137.27
   7 |    3,184,586.42 |  -19,130,550.85
   8 |    2,948,691.13 |  -16,181,859.72
   9 |    2,730,269.56 |  -13,451,590.16
  10 |    2,528,027.37 |  -10,923,562.79
  11 |    2,340,766.09 |   -8,582,796.70
  12 |    2,167,376.01 |   -6,415,420.69
  13 |    2,006,829.64 |   -4,408,591.06
  14 |    1,858,175.59 |   -2,550,415.47
  15 |    1,720,532.95 |     -829,882.52
  16 |    1,457,019.88 |      627,137.36
  17 |    1,349,092.48 |    1,976,229.84
  18 |    1,249,159.71 |    3,225,389.55
  19 |    1,156,629.36 |    4,382,018.90
  20 |    

### **Step 5 - Update Parameters as Necessary**

As the other groups provide us with more detailed data, we can update our calculations as necessary.
Let's use the example of "what if the discount rate was higher"?

- **Discount Rate**: The "interest rate" applied to calculate the NPV of a project's future cash flows - determines how viable a project is in the long-term.

In [11]:
# Scenario: what if the discount rate rises to 12 %?
print(f"NPV at  8 % discount rate: ${plant.calculate_npv():>14,.0f}")

plant.update_configuration({"interest_rate": 0.12})
plant.calculate_all()
print(f"NPV at 12 % discount rate: ${plant.calculate_npv():>14,.0f}")

# reset
plant.update_configuration({"interest_rate": 0.08})
plant.calculate_all()

NPV at  8 % discount rate: $     6,210,963
NPV at 12 % discount rate: $    -4,587,729


### **Step 6 - Override "Default" Parameters**

Depending on specific project estimates, we can override default factors for fixed capital costs (contingency, engineering, ...) and fixed OPEX (labour costs, working days per year, ...)
For the time being, we will rely on defaults, but as the simulation starts to form and we get better real-life estimates, we can start replacing default factors. 

Example: what if we were extra careful with maintaining the plant, and increased budgets for that procedure?

In [12]:
# Default fixed OPEX
print(f"\nDefault Fixed OPEX : ${plant.fixed_production_costs:>14,.0f} / yr")

# Override: higher maintenance factor + direct maintenance budget
plant.update_configuration({
    "fixed_opex_factors": {
        "maintenance": 0.07,           # 7 % of ISBL instead of 5 %
        "laboratory_charges": 0.12,    # 12 % of operating labour instead of 10 %
    },
    "fixed_opex_components": {
        "rent_of_land_costs": 100_000,  # fixed annual rent, USD/yr
    },
})

plant.calculate_fixed_opex(print_results=True)
print(plant.fixed_production_costs)


Default Fixed OPEX : $     5,222,183 / yr
Fixed production costs estimation
Operating labor costs: 893,760.00 USD per year
Supervision costs: 223,440.00 USD per year
Direct salary overhead: 558,600.00 USD per year
Laboratory charges: 107,251.20 USD per year
Maintenance costs: 905,888.16 USD per year
Taxes and insurance costs: 194,118.89 USD per year
Rent of land costs: 100,000.00 USD per year
Environmental charges: 168,236.37 USD per year
Operating supplies: 116,471.34 USD per year
General plant overhead: 1,089,270.00 USD per year
Interest on working capital: 282,637.11 USD per year
Patents and royalties: 204,633.00 USD per year
Distribution and selling costs: 204,633.00 USD per year
R&D costs: 306,949.49 USD per year
Fixed OPEX: 5,355,888.56 USD per year
5355888.557917053


### **Step 7 - Calculate Cash Flow!**

It's time to evaluate the long-term benefit for investors!

`calculate_cash_flow()` builds a year-by-year cash flow table covering CAPEX spend, revenue, operating costs, depreciation, taxes, and net cash flow. It returns a `pd.DataFrame` (or `None` if `print_results=True`). 

Two optional ramp profiles control the timing:

| Config key | Default | Description |
|---|---|---|
| `capex_ramp` | `[0.3, 0.6, 0.1]` | Fraction of FCI spent in each construction year (must sum to 1) |
| `production_ramp` | `[0, 0, 0.4, 0.8]` | Nameplate capacity fraction for each project year |

In [13]:
# Generate cash flow with default ramp profiles
plant.calculate_cash_flow(print_results=True)

,Year,Capital cost [USD],Revenue [USD],Cash cost [USD],Gross profit [USD],Depreciation [USD],Taxable income [USD],Tax paid [USD],Cash flow [USD]
0,1,"7,065,927.68",0.00,"5,355,888.56","-5,355,888.56",0.00,"-5,355,888.56",0.00,"-12,421,816.23"
1,2,"14,131,855.35",0.00,"5,355,888.56","-5,355,888.56",0.00,"-5,355,888.56",0.00,"-19,487,743.91"
2,3,"5,888,273.06","6,701,400.00","7,306,193.06","-604,793.06","1,864,619.80","-2,469,412.86",0.00,"-6,493,066.12"
3,4,0.00,"13,402,800.00","9,256,497.56","4,146,302.44","1,864,619.80","2,281,682.64",0.00,"4,146,302.44"
4,5,0.00,"16,753,500.00","10,231,649.81","6,521,850.19","1,864,619.80","4,657,230.39","570,420.66","5,951,429.53"
5,6,0.00,"16,753,500.00","10,231,649.81","6,521,850.19","1,864,619.80","4,657,230.39","1,164,307.60","5,357,542.59"
6,7,0.00,"16,753,500.00","10,231,649.81","6,521,850.19","1,864,619.80","4,657,230.39","1,164,307.60","5,357,542.59"
7,8,0.00,"16,753,500.00","10,231,649.81","6,521,850.19","1,864,619.80","4,657,230.39","1,164,307.60","5,357,542.59"
8,9,0.00,"16,753,500.00","10,231,649.81","6,521,850.19","1,864,619.80","4,657,230.39","1,164,307.60","5,357,542.59"
9,10,0.00,"16,753,500.00","10,231,649.81","6,521,850.19","1,864,619.80","4,657,230.39","1,164,307.60","5,357,542.59"


### **Step 8 - Modifying Depreciation Model**

We can fine-tune the depreciation of the plant (loss of efficiency through time) using different models.

| Method | Key | Required fields |
|---|---|---|
| Straight-line | `"straight_line"` | `life`, `salvage_fraction`, `service_start_year` |
| Declining balance | `"declining_balance"` | `life`, `salvage_fraction`, `db_factor`, `service_start_year` |
| MACRS | `"macrs"` | `macrs_class` |

Switch between methods via `update_configuration()` — the new config replaces the previous depreciation block entirely.

### **Step 9 (Future Action) - Use Charts to Compare Different Plants**

This live demo showcases how to calculate TEA for one sample plants.
To save time, we will directly browse case study for a real-life example of how to compare different plants using charts and additional functions!

## **Key Takeaways**

- Plenty of defaults allow us to start prototyping very fast with a low number of parameters.
- All default parameters are customisable, so we can update the calculations constantly as new data comes in.
- We do not need to wait! Let's communicate between groups!
- If appropriate, we can design a GitHub repository every group can access to check and update current calculations.

## **Summary**

- Designing and evaluating a process plant simulation in OpenPyTEA has 8 steps.
- We follow the basic procedure of: Equipment => Plant (Factory) => Cost-Benefit / TEA Evaluation
- Once we have information on multiple processes/"factories", we can compare them together in terms of CAPEX, OPEX, energy consumption, levelised costs, ...